# Extract HTML review data

Run the cells in order after `data_cleaning.ipynb`, then run
`merge_data_and_text.ipynb` to link the reviews to the cleaned data.

This notebook performs the work of `scripts/html_to_txt.py` and
`scripts/txt_to_csv.py`. All processing code is included here and uses only
Python's standard library. The Python files are kept as a separate way to run
this pipeline.

The outputs are:

- `outputs/text_reviews/<source_id>.txt`: description, a blank line, then review paragraphs.
- `outputs/review_text.csv`: the eight metadata and text columns needed for linking.
- `outputs/extracting_text/`: the HTML/text manifest, duplicate report,
  missing-description list, and skipped-HTML log.

Names retain their original punctuation. A source ID is derived from the
canonical review URL. Every member of duplicate source-ID or normalized
artist/album/date groups is discarded. Text files without descriptions are
retained and reported, but excluded from `review_text.csv`.

Saving the extracted text archives the previous text directory before replacing
it. Raw HTML and `data_clean.csv` are preserved.

## 1. Imports and paths

Edit `HTML_INPUTS` to process specific files or directories. Leave the defaults
to process the full collection. The input directories are searched recursively.

In [1]:
from collections import Counter, defaultdict
import csv
from datetime import date
from decimal import Decimal, InvalidOperation
import hashlib
from html.parser import HTMLParser
import json
from pathlib import Path
from pprint import pprint
import re
import sys
import tempfile
import unicodedata
from urllib.parse import urlsplit, urlunsplit


In [2]:
# Locate the repository from its root, notebooks/, or a deeper directory.
REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data.csv").is_file() and (path / "notebooks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook with this repository as the working directory.")

OUTPUTS_DIR = REPO_ROOT / "outputs"
EXTRACTION_DIR = OUTPUTS_DIR / "extracting_text"

HTML_INPUTS = [REPO_ROOT / "pitchfork-scrape" / "raw-html"]
TEXT_OUTPUT_DIR = OUTPUTS_DIR / "text_reviews"
REVIEW_CSV = OUTPUTS_DIR / "review_text.csv"

print("Repository:", REPO_ROOT)
print("HTML inputs:", HTML_INPUTS)
print("Text directory:", TEXT_OUTPUT_DIR)
print("Support directory:", EXTRACTION_DIR)
print("Metadata CSV:", REVIEW_CSV)


Repository: /home/ai4covid/test_stuff_2
HTML inputs: [PosixPath('/home/ai4covid/test_stuff_2/pitchfork-scrape/raw-html')]
Text directory: /home/ai4covid/test_stuff_2/outputs/text_reviews
Support directory: /home/ai4covid/test_stuff_2/outputs/extracting_text
Metadata CSV: /home/ai4covid/test_stuff_2/outputs/review_text.csv


## 2. CSV schemas and matching rules

The metadata CSV contains only `source_id`, `artist`, `album`, `reviewdate`,
`reviewauthor`, `score`, `description`, and `review`. The separate manifest
also records the HTML and text filenames.

Matching normalization handles Unicode, curly quotes, whitespace, and case.
It does not change the names saved in the CSV or remove punctuation.

In [3]:
IDENTITY_FIELDS = [
    "source_id", "artist", "album", "reviewdate", "reviewauthor", "score"
]


REVIEW_FIELDS = IDENTITY_FIELDS + ["description", "review"]


MAP_FIELDS = ["source_id", "html_file_name", "text_file_name"] + IDENTITY_FIELDS[1:]


In [4]:
QUOTE_MAP = str.maketrans({
    "‘": "'", "’": "'", "‛": "'", "′": "'", "“": '"', "”": '"', "″": '"',
})


def normalize_name(value):
    """Normalize Unicode, quotes, whitespace, and case without losing punctuation."""
    return " ".join(
        unicodedata.normalize("NFKC", value or "").translate(QUOTE_MAP).split()
    ).casefold()


def review_date(value):
    return date.fromisoformat((value or "").strip()[:10]).isoformat()


def review_score(value):
    try:
        score = Decimal((value or "").strip())
    except InvalidOperation as exc:
        raise ValueError(f"invalid review score: {value!r}") from exc
    if not score.is_finite() or not 0 <= score <= 10:
        raise ValueError(f"invalid review score: {value!r}")
    return score


def match_key(row):
    artist, album = (normalize_name(row.get(field)) for field in ("artist", "album"))
    if not artist or not album:
        raise ValueError("missing artist or album")
    return artist, album, review_date(row.get("reviewdate"))


def duplicate_keys(rows):
    counts = Counter()
    for row in rows:
        try:
            counts[match_key(row)] += 1
        except ValueError:
            continue
    return {key for key, count in counts.items() if count > 1}


In [5]:
def canonical_url(value):
    parts = urlsplit((value or "").strip())
    if parts.hostname not in {"pitchfork.com", "www.pitchfork.com"}:
        raise ValueError("missing or invalid canonical Pitchfork URL")
    if not parts.path.startswith("/reviews/albums/"):
        raise ValueError("canonical URL is not an album review")
    return urlunsplit(("https", "pitchfork.com", parts.path.rstrip("/") + "/", "", ""))


def source_id_for_url(value):
    return hashlib.sha256(canonical_url(value).encode("utf-8")).hexdigest()[:32]


def read_csv(path, required=()):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        fields = reader.fieldnames or []
        missing = set(required) - set(fields)
        if missing:
            raise ValueError(f"{path}: missing columns {sorted(missing)}")
        if len(set(fields)) != len(fields):
            raise ValueError(f"{path}: duplicate column names")
        return fields, list(reader)


def write_csv(path, fields, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


## 3. Parse the original HTML

Read the first artist link and first album heading, rather than the combined
HTML title. Publication date and author come from JSON-LD; score comes from the
first score element. The meta description is extracted separately from the
review paragraphs. Invalid or incomplete pages are recorded in the skipped log.

In [6]:
class ReviewParser(HTMLParser):
    """Read visible identity fields, JSON-LD, description, and review paragraphs."""

    VOID_TAGS = {
        "area", "base", "br", "col", "embed", "hr", "img", "input", "link",
        "meta", "param", "source", "track", "wbr",
    }
    IGNORED_TAGS = {"script", "style", "noscript", "iframe", "svg"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.stack = []
        self.values = {}
        self.captures = {}
        self.description = ""
        self.url = ""
        self.og_url = ""
        self.json_blocks = []
        self.json_parts = None
        self.json_depth = None
        self.artist_list_depth = None
        self.paragraphs = []
        self.paragraph_parts = None
        self.review_depth = None

    def capture(self, name):
        if name not in self.values and name not in self.captures:
            self.captures[name] = (len(self.stack), [])

    def finish_paragraph(self):
        if self.paragraph_parts is not None:
            paragraph = " ".join("".join(self.paragraph_parts).split())
            if paragraph:
                self.paragraphs.append(paragraph)
            self.paragraph_parts = None

    def handle_starttag(self, tag, attrs):
        ignored = any(name in self.IGNORED_TAGS for name in self.stack)
        attrs = dict(attrs)
        if not ignored:
            if tag == "meta":
                name = (attrs.get("name") or attrs.get("property") or "").lower()
                if name == "description" and not self.description:
                    self.description = " ".join((attrs.get("content") or "").split())
                elif name == "og:url":
                    self.og_url = attrs.get("content") or ""
            if tag == "link" and "canonical" in (attrs.get("rel") or "").split():
                self.url = attrs.get("href") or ""
        if tag == "br" and not ignored:
            if self.paragraph_parts is not None:
                self.paragraph_parts.append(" ")
            for _, parts in self.captures.values():
                parts.append(" ")
        if tag in self.VOID_TAGS:
            return
        self.stack.append(tag)
        if ignored:
            return
        classes = (attrs.get("class") or "").split()
        if tag == "script" and attrs.get("type") == "application/ld+json":
            self.json_parts = []
            self.json_depth = len(self.stack)
        if tag == "ul" and "artist-links" in classes and "artist" not in self.values:
            self.artist_list_depth = len(self.stack)
        if tag == "li" and self.artist_list_depth is not None:
            self.capture("artist")
        if tag == "h1" and "single-album-tombstone__review-title" in classes:
            self.capture("album")
        if tag == "span" and "score" in classes:
            self.capture("score")
        if tag == "div" and "contents" in classes and self.review_depth is None:
            self.review_depth = len(self.stack)
        if tag == "p" and self.review_depth is not None:
            self.finish_paragraph()
            self.paragraph_parts = []

    def handle_endtag(self, tag):
        if tag not in self.stack:
            return
        index = len(self.stack) - 1 - self.stack[::-1].index(tag)
        ignored = any(name in self.IGNORED_TAGS for name in self.stack)
        for name, (depth, parts) in list(self.captures.items()):
            if index < depth:
                value = " ".join("".join(parts).split())
                # An empty first heading must not select a later album's name.
                self.values[name] = value
                del self.captures[name]
        if self.json_depth is not None and index < self.json_depth:
            self.json_blocks.append("".join(self.json_parts))
            self.json_parts = self.json_depth = None
        if self.artist_list_depth is not None and index < self.artist_list_depth:
            self.artist_list_depth = None
        if tag == "p" and not ignored:
            self.finish_paragraph()
        if self.review_depth is not None and index < self.review_depth:
            self.finish_paragraph()
            self.review_depth = None
        del self.stack[index:]

    def handle_data(self, data):
        if self.json_parts is not None:
            self.json_parts.append(data)
            return
        if any(tag in self.IGNORED_TAGS for tag in self.stack):
            return
        for _, parts in self.captures.values():
            parts.append(data)
        if self.paragraph_parts is not None:
            self.paragraph_parts.append(data)


In [7]:
def json_objects(value):
    """Support plain JSON-LD objects, lists, and @graph documents."""
    if isinstance(value, list):
        for item in value:
            yield from json_objects(item)
    elif isinstance(value, dict):
        yield value
        if "@graph" in value:
            yield from json_objects(value["@graph"])


def extract_review(html):
    parser = ReviewParser()
    parser.feed(html)
    parser.close()
    parser.finish_paragraph()
    if not parser.paragraphs:
        raise ValueError("no review paragraphs found in a div.contents container")
    for field in ("artist", "album", "score"):
        if not parser.values.get(field):
            raise ValueError(f"no nonempty {field} field found in the review heading")
    article = None
    for block in parser.json_blocks:
        try:
            candidates = json_objects(json.loads(block))
            article = next((obj for obj in candidates if obj.get("datePublished")), None)
        except json.JSONDecodeError:
            continue
        if article:
            break
    if not article:
        raise ValueError("no publication metadata found in JSON-LD")
    author = article.get("author") or article.get("creator") or {}
    if isinstance(author, list):
        author = author[0] if len(author) == 1 else {}
    author = author.get("name", "") if isinstance(author, dict) else author
    author = " ".join((author or "").split())
    if not author:
        raise ValueError("no review author found")
    url = canonical_url(parser.url or parser.og_url or article.get("url"))
    return {
        "source_id": source_id_for_url(url),
        "artist": parser.values["artist"],
        "album": parser.values["album"],
        "reviewdate": review_date(article.get("datePublished")),
        "reviewauthor": author,
        "score": str(review_score(parser.values["score"])),
        "description": parser.description,
        "review": "\n\n".join(parser.paragraphs) + "\n",
    }


## 4. Find and extract reviews

The complete collection can take a few minutes. Progress is printed every
2,000 files. No existing output is replaced during this parsing step.

In [8]:
files = set()
for source in map(Path, HTML_INPUTS):
    if source.is_dir():
        files.update(
            path.resolve() for path in source.rglob("*")
            if path.is_file() and path.suffix.lower() in {".html", ".htm"}
        )
    elif source.is_file():
        files.add(source.resolve())
    else:
        raise FileNotFoundError(f"Input does not exist: {source}")
if not files:
    raise ValueError("No HTML files found.")
print(f"Found {len(files):,} HTML files.")


Found 21,321 HTML files.


In [9]:
records, skipped = [], []
for index, source in enumerate(sorted(files), 1):
    try:
        record = extract_review(source.read_text(encoding="utf-8-sig"))
        records.append({
            **record,
            "html_file_name": source.name,
            "text_file_name": record["source_id"] + ".txt",
        })
    except (OSError, UnicodeError, ValueError) as exc:
        skipped.append(f"{source}\t{exc}\n")
        print(f"Skipping {source.name}: {exc}", file=sys.stderr)
    if index % 2000 == 0:
        print(f"Parsed {index}/{len(files)} HTML files.", flush=True)
if not records:
    raise ValueError("No reviews could be extracted; previous outputs were preserved.")

print(f"Parsed {len(records):,} reviews; skipped {len(skipped):,} source files.")
for row in records[:3]:
    pprint({field: row[field] for field in IDENTITY_FIELDS})


Skipping 11617-partie-t.html: no review paragraphs found in a div.contents container


Parsed 2000/21321 HTML files.
Parsed 4000/21321 HTML files.


Skipping 1391-being-ridden.html: no review paragraphs found in a div.contents container


Parsed 6000/21321 HTML files.


Skipping 16697-record.html: no nonempty album field found in the review heading


Parsed 8000/21321 HTML files.
Parsed 10000/21321 HTML files.
Parsed 12000/21321 HTML files.
Parsed 14000/21321 HTML files.
Parsed 16000/21321 HTML files.


Skipping 8150-triang.html: no nonempty album field found in the review heading


Parsed 18000/21321 HTML files.


Skipping 9464-shine-on.html: no review paragraphs found in a div.contents container


Parsed 20000/21321 HTML files.
Parsed 21,316 reviews; skipped 5 source files.
{'album': 'TFCF',
 'artist': 'Liars',
 'reviewauthor': 'Patric Fallon',
 'reviewdate': '2017-08-28',
 'score': '7.5',
 'source_id': 'c7af38b5e299e7cdece3756934c1e947'}
{'album': 'Connected and Respected',
 'artist': 'E-40',
 'reviewauthor': 'Torii MacAdams',
 'reviewdate': '2018-04-14',
 'score': '6.7',
 'source_id': '03b726ca0a9b1a25d770fb7a42cdebfc'}
{'album': '40oz. to Freedom',
 'artist': 'Sublime',
 'reviewauthor': 'Evan Rytlewski',
 'reviewdate': '2018-02-25',
 'score': '5.6',
 'source_id': '22cf35b500b7ee65b2b0f2f9988a748b'}


## 5. Exclude duplicate HTML groups and save text files

Discard all copies of a repeated source ID or normalized artist/album/date
key. Stage the new text files before archiving the previous text directory.
Write all four support files to `outputs/extracting_text/`.

In [10]:
def write_outputs(records, skipped, output_dir, reports_dir):
    """Discard entire duplicate groups and replace the text directory together."""
    duplicate_ids = {
        value for value, count in Counter(row["source_id"] for row in records).items()
        if count > 1
    }
    duplicated_keys = duplicate_keys(records)
    kept, discarded = [], []
    for record in records:
        if record["source_id"] in duplicate_ids or match_key(record) in duplicated_keys:
            discarded.append({
                **{field: record[field] for field in MAP_FIELDS},
                "discard_reason": "duplicate_html_review",
            })
        else:
            kept.append(record)

    output_dir.parent.mkdir(parents=True, exist_ok=True)
    backup = None
    with tempfile.TemporaryDirectory(prefix=".review_stage_", dir=output_dir.parent) as tmp:
        stage = Path(tmp) / "text_reviews"
        stage.mkdir()
        for record in kept:
            text = record["review"]
            if record["description"]:
                text = record["description"] + "\n\n" + text
            (stage / record["text_file_name"]).write_text(text, encoding="utf-8")
        if output_dir.exists():
            backup = Path(tempfile.mkdtemp(prefix="previous_text_reviews_", dir=output_dir.parent))
            output_dir.rename(backup / "text_reviews")
        try:
            stage.rename(output_dir)
        except OSError:
            if backup is not None:
                (backup / "text_reviews").rename(output_dir)
            raise

    write_csv(
        reports_dir / "html_text_match.csv", MAP_FIELDS,
        [{field: row[field] for field in MAP_FIELDS} for row in kept],
    )
    write_csv(reports_dir / "discarded_html.csv", MAP_FIELDS + ["discard_reason"], discarded)
    (reports_dir / "missing_descriptions.txt").write_text(
        "".join(row["text_file_name"] + "\n" for row in kept if not row["description"]),
        encoding="utf-8",
    )
    (reports_dir / "skipped_htmls.txt").write_text("".join(skipped), encoding="utf-8")
    return len(kept), len(discarded), backup


In [11]:
converted, discarded, backup = write_outputs(
    records, skipped, TEXT_OUTPUT_DIR, EXTRACTION_DIR
)
print(f"Wrote {converted:,} text files.")
print(f"Discarded {discarded:,} duplicate HTML files; skipped {len(skipped):,} pages.")
if backup is not None:
    print("Previous text files archived in:", backup)
print("Manifest and extraction reports:", EXTRACTION_DIR)


Wrote 21,312 text files.
Discarded 4 duplicate HTML files; skipped 5 pages.
Manifest and extraction reports: /home/ai4covid/test_stuff_2/outputs/extracting_text


## 6. Build the review metadata CSV

Read identity fields from the manifest, never from the filename. Exclude the
source-ID text filenames in `missing_descriptions.txt`. Split each remaining
text file at the first blank line: the first paragraph is the description and
all subsequent paragraphs form the review.

All input rows are validated before replacing `review_text.csv`.

In [12]:
def review_row(path, metadata):
    text = path.read_text(encoding="utf-8-sig").strip()
    paragraphs = re.split(r"\n[ \t]*\n", text, maxsplit=1)
    if len(paragraphs) != 2 or not all(part.strip() for part in paragraphs):
        raise ValueError("expected a description followed by review paragraphs")
    return {
        **{field: metadata[field] for field in IDENTITY_FIELDS},
        "description": paragraphs[0].strip(), "review": paragraphs[1].strip(),
    }


In [13]:
if not TEXT_OUTPUT_DIR.is_dir():
    raise FileNotFoundError(f"Text directory does not exist: {TEXT_OUTPUT_DIR}")
_, manifest = read_csv(EXTRACTION_DIR / "html_text_match.csv", MAP_FIELDS)
excluded = set(
    (EXTRACTION_DIR / "missing_descriptions.txt").read_text(encoding="utf-8-sig").splitlines()
)
csv_rows, seen = [], set()
missing_description_count = 0
for metadata in manifest:
    source_id = metadata["source_id"]
    name = metadata["text_file_name"]
    if not re.fullmatch(r"[0-9a-f]{32}", source_id) or name != source_id + ".txt":
        raise ValueError(f"Invalid source ID/text filename in manifest: {name}")
    if source_id in seen:
        raise ValueError(f"Duplicate source ID in manifest: {source_id}")
    seen.add(source_id)
    if name in excluded:
        missing_description_count += 1
        continue
    csv_rows.append(review_row(TEXT_OUTPUT_DIR / name, metadata))

assert len(manifest) == len(csv_rows) + missing_description_count
assert len({row["source_id"] for row in csv_rows}) == len(csv_rows)
print(f"Prepared {len(csv_rows):,} CSV rows; excluded {missing_description_count:,} without descriptions.")


Prepared 21,271 CSV rows; excluded 41 without descriptions.


In [14]:
write_csv(REVIEW_CSV, REVIEW_FIELDS, csv_rows)
extraction_summary = {
    "html_files": len(files),
    "text_files": converted,
    "duplicate_html_files_discarded": discarded,
    "html_files_skipped": len(skipped),
    "missing_descriptions": missing_description_count,
    "review_csv_rows": len(csv_rows),
}
pprint(extraction_summary)
print("Saved:", REVIEW_CSV)


{'duplicate_html_files_discarded': 4,
 'html_files': 21321,
 'html_files_skipped': 5,
 'missing_descriptions': 41,
 'review_csv_rows': 21271,
 'text_files': 21312}
Saved: /home/ai4covid/test_stuff_2/outputs/review_text.csv


## Next step

Run `merge_data_and_text.ipynb` to link `outputs/review_text.csv` to
`outputs/data_clean.csv`, with author and score checks and separate reports for
unresolved and discarded rows.